# Phase 8: Optional 3D OSP-Regularized MSFA

Goal: upgrade the sphere-packing prior from a spectral-centroid-only surrogate to a spatio-spectral 3D surrogate.

Practical note:
- This notebook is optional.
- Keep the 1D SP result as fallback.
- Run this only after Phase 4 is already working.


# Learnable MSFA Research Track

This notebook is part of the 10-day publication-oriented extension track.

## Run discipline
- Keep one experiment purpose per notebook.
- Save metrics and checkpoints to the configured project root.
- Do not silently change data splits or loss definitions across notebooks.
- When a result becomes final, export both numeric artifacts and a figure/table asset.


In [ ]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    print("Google Drive mounted.")
except Exception as exc:
    print("Drive mount skipped:", exc)


In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

PROJECT_ROOT = Path("/content/drive/MyDrive/Msa-Osp")
PATCH_PATH = PROJECT_ROOT / "dataset_patches.npz"
INIT_CKPT_PATH = PROJECT_ROOT / "learned_msfa_sp_best.pth"
HISTORY_PATH = PROJECT_ROOT / "learned_msfa_3dsp_history.csv"
CKPT_PATH = PROJECT_ROOT / "learned_msfa_3dsp_best.pth"
TILE_PATH = PROJECT_ROOT / "learned_msfa_3dsp_tile_soft.npy"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 8
EPOCHS = 30
UNET_LR = 1e-4
MSFA_LR = 2e-4
TEMP = 0.05
BAND_COUNT = 16
TILE_SIZE = 4
LAMBDA_SP = 5e-4
LAMBDA_SMOOTH = 1e-4
D_MIN_3D = 0.55
Z_WEIGHT = 1.25
WAVELENGTHS = torch.linspace(0.0, 1.0, BAND_COUNT)

print("Warm-start checkpoint exists:", INIT_CKPT_PATH.exists())


In [ ]:
data = np.load(PATCH_PATH)
train_target = data["train"]
val_target = data["val"]

class CubeDataset(Dataset):
    def __init__(self, cubes):
        self.cubes = cubes

    def __len__(self):
        return len(self.cubes)

    def __getitem__(self, idx):
        return torch.from_numpy(self.cubes[idx]).permute(2, 0, 1).float()

train_loader = DataLoader(CubeDataset(train_target), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(CubeDataset(val_target), batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class UNet2D(nn.Module):
    def __init__(self, in_ch=1, out_ch=16, base=64):
        super().__init__()
        self.enc1 = DoubleConv(in_ch, base)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = DoubleConv(base, base * 2)
        self.pool2 = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(base * 2, base * 4)
        self.up2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
        self.dec2 = DoubleConv(base * 4, base * 2)
        self.up1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
        self.dec1 = DoubleConv(base * 2, base)
        self.final = nn.Conv2d(base, out_ch, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        b = self.bottleneck(self.pool2(e2))
        d2 = self.dec2(torch.cat([self.up2(b), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.final(d1)

class LearnableMSFA(nn.Module):
    def __init__(self, bands=BAND_COUNT, tile_size=TILE_SIZE, temperature=TEMP):
        super().__init__()
        self.tile_size = tile_size
        self.temperature = temperature
        self.logits = nn.Parameter(torch.randn(bands, tile_size, tile_size) * 0.01)

    def soft_tile(self):
        return torch.softmax(self.logits / self.temperature, dim=0)

    def forward(self, x):
        _, _, h, w = x.shape
        weights = self.soft_tile()
        weights_full = weights.repeat(1, h // self.tile_size, w // self.tile_size)
        return (x * weights_full.unsqueeze(0)).sum(dim=1, keepdim=True)

def compute_psnr(pred, target, eps=1e-8):
    mse = torch.mean((pred - target) ** 2)
    return 20 * torch.log10(1.0 / torch.sqrt(mse + eps))

def spectral_angle_mapper(pred, target, eps=1e-8):
    dot = torch.sum(pred * target, dim=1)
    pred_norm = torch.norm(pred, dim=1)
    target_norm = torch.norm(target, dim=1)
    cos_theta = torch.clamp(dot / (pred_norm * target_norm + eps), -1 + eps, 1 - eps)
    return torch.rad2deg(torch.acos(cos_theta)).mean()

def spectral_smoothness_loss(msfa):
    weights = msfa.soft_tile()
    return torch.mean((weights[1:] - weights[:-1]) ** 2)


In [ ]:
spatial_coords = torch.stack(
    torch.meshgrid(
        torch.linspace(0.0, 1.0, TILE_SIZE),
        torch.linspace(0.0, 1.0, TILE_SIZE),
        indexing="ij",
    ),
    dim=-1,
).reshape(-1, 2)

def spatio_spectral_points(msfa):
    weights = msfa.soft_tile()
    z = (weights * WAVELENGTHS.to(weights.device).view(BAND_COUNT, 1, 1)).sum(dim=0).reshape(-1, 1)
    xy = spatial_coords.to(weights.device)
    return torch.cat([xy, Z_WEIGHT * z], dim=1)

def pairwise_3d_distances(msfa):
    points = spatio_spectral_points(msfa)
    d = torch.cdist(points, points, p=2)
    mask = torch.triu(torch.ones_like(d), diagonal=1) > 0
    return d[mask]

def osp_3d_loss(msfa, d_min=D_MIN_3D):
    distances = pairwise_3d_distances(msfa)
    return torch.relu(d_min - distances).mean()

def distance_stats(msfa):
    distances = pairwise_3d_distances(msfa)
    return {
        "min_3d_distance": distances.min().item(),
        "mean_3d_distance": distances.mean().item(),
    }


In [ ]:
unet = UNet2D().to(DEVICE)
msfa = LearnableMSFA().to(DEVICE)

if INIT_CKPT_PATH.exists():
    checkpoint = torch.load(INIT_CKPT_PATH, map_location=DEVICE)
    unet.load_state_dict(checkpoint["unet"])
    msfa.load_state_dict(checkpoint["msfa"])
    print("Warm-started from 1D SP checkpoint.")
else:
    print("Starting from scratch.")

optimizer = torch.optim.Adam(
    [
        {"params": unet.parameters(), "lr": UNET_LR},
        {"params": msfa.parameters(), "lr": MSFA_LR},
    ]
)
criterion = nn.L1Loss()
best_psnr = -float("inf")
history = []

for epoch in range(1, EPOCHS + 1):
    unet.train()
    msfa.train()
    train_loss = 0.0
    train_psnr = 0.0

    for x in train_loader:
        x = x.to(DEVICE)
        pred = unet(msfa(x))
        recon_loss = criterion(pred, x)
        sp_loss = osp_3d_loss(msfa)
        smooth_loss = spectral_smoothness_loss(msfa)
        loss = recon_loss + LAMBDA_SP * sp_loss + LAMBDA_SMOOTH * smooth_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        train_psnr += compute_psnr(pred, x).item()

    unet.eval()
    msfa.eval()
    val_psnr = 0.0
    val_sam = 0.0
    with torch.no_grad():
        for x in val_loader:
            x = x.to(DEVICE)
            pred = unet(msfa(x))
            val_psnr += compute_psnr(pred, x).item()
            val_sam += spectral_angle_mapper(pred, x).item()

    train_loss /= len(train_loader)
    train_psnr /= len(train_loader)
    val_psnr /= len(val_loader)
    val_sam /= len(val_loader)
    sp_value = osp_3d_loss(msfa).item()
    stats = distance_stats(msfa)
    history.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "train_psnr": train_psnr,
            "val_psnr": val_psnr,
            "val_sam_deg": val_sam,
            "osp_3d_loss": sp_value,
            "min_3d_distance": stats["min_3d_distance"],
            "mean_3d_distance": stats["mean_3d_distance"],
        }
    )
    print(
        f"Epoch {epoch:02d} | train PSNR {train_psnr:.2f} dB | "
        f"val PSNR {val_psnr:.2f} dB | val SAM {val_sam:.2f} deg | "
        f"3D sp {sp_value:.4f} | min 3D d {stats['min_3d_distance']:.3f}"
    )

    if val_psnr > best_psnr:
        best_psnr = val_psnr
        CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)
        torch.save(
            {
                "epoch": epoch,
                "unet": unet.state_dict(),
                "msfa": msfa.state_dict(),
                "best_val_psnr": best_psnr,
                "lambda_sp": LAMBDA_SP,
                "lambda_smooth": LAMBDA_SMOOTH,
                "d_min_3d": D_MIN_3D,
                "z_weight": Z_WEIGHT,
                "temperature": TEMP,
                "distance_stats": stats,
            },
            CKPT_PATH,
        )
        TILE_PATH.parent.mkdir(parents=True, exist_ok=True)
        np.save(TILE_PATH, msfa.soft_tile().detach().cpu().numpy())

HISTORY_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(HISTORY_PATH, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(history[0].keys()))
    writer.writeheader()
    writer.writerows(history)


In [ ]:
checkpoint = torch.load(CKPT_PATH, map_location=DEVICE)
msfa.load_state_dict(checkpoint["msfa"])

soft_tile = msfa.soft_tile().detach().cpu().numpy()
hard_tile = soft_tile.argmax(axis=0)
centroid_nm = (soft_tile * np.linspace(400.0, 700.0, BAND_COUNT)[:, None, None]).sum(axis=0)

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.imshow(hard_tile, cmap="tab20")
plt.title("Hard tile after 3D SP training")
plt.colorbar()

plt.subplot(1, 2, 2)
plt.imshow(centroid_nm, cmap="viridis")
plt.title("Centroid wavelengths (nm)")
plt.colorbar(label="nm")
plt.tight_layout()
plt.show()

print("Best 3D distance stats:", checkpoint.get("distance_stats", {}))


If this 3D variant does not clearly beat the 1D SP result, keep the 1D SP result as the main paper line and report the 3D version as an exploratory extension.
